In [2]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib as mpl
font = {'family' : 'normal',
        'weight' : 'bold',
        'size'   : 12}

plt.rc('font', **font)
plot_fs = 12

plt.rc('font', family='serif', serif='Times')
plt.rc('text', usetex=True)
plt.rc('xtick', labelsize=9)
plt.rc('ytick', labelsize=9)
plt.rc('axes', labelsize=10)
mpl.rcParams['lines.dashed_pattern'] = [2, 2]
mpl.rcParams['lines.linewidth'] = 1.0

import pathlib
cur_dir = pathlib.Path().cwd()
parent_dir = cur_dir.parent

import sys
print(sys.version)
sys.path.append('../')

from models.powertrain.bounded_powertrain import Bounded_powertrain
from models.kinematic.ideal_diff_drive import Ideal_diff_drive
from models.learning.blr_slip import SlipBayesianLinearRegression, FullBodySlipBayesianLinearRegression
from models.learning.blr_slip_acceleration import SlipAccelerationBayesianLinearRegression, FullBodySlipAccelerationBayesianLinearRegression
from models.kinematic.ICR_based import *
from models.kinematic.Perturbed_unicycle import *
from models.kinematic.enhanced_kinematic import *
from NicolasSamson.script.extractors import * 
from util.transform_algebra import *
from util.util_func import *


3.8.19 (default, Apr  6 2024, 17:58:10) 
[GCC 11.4.0]


In [3]:
# import slip dataset

# dataset_path = '../data/ral2023_dataset/husky/boreal_snow/slip_dataset_all.pkl'
# dataset_path = '../data/ral2023_dataset/husky/grand_salon_tile_inflated/slip_dataset_all.pkl'
# dataset_path = '../data/ral2023_dataset/husky/grand_salon_left-deflated/slip_dataset_all.pkl'
# dataset_path = '../data/ral2023_dataset/marmotte/boreal_snow/slip_dataset_all.pkl'
# dataset_path = '../data/ral2023_dataset/marmotte/ga_hard_snow_a/slip_dataset_all.pkl'
# dataset_path = '../data/ral2023_dataset/marmotte/ga_hard_snow_b/slip_dataset_all.pkl'
# dataset_path = '../data/ral2023_dataset/marmotte/grand_salon_tile_b/slip_dataset_all.pkl'
# dataset_path = '../data/ral2023_dataset/warthog_tracks/boreal_mud/slip_dataset_all.pkl'
# dataset_path = '../data/ral2023_dataset/warthog_tracks/grand-axe_crusted-snow/slip_dataset_all.pkl'

# dataset_path = '../data/ral2023_dataset/husky/boreal_snow/acceleration_dataset.pkl'
# dataset_path = '../data/ral2023_dataset/husky/grand_salon_tile_inflated/acceleration_dataset.pkl'
# dataset_path = '../data/ral2023_dataset/husky/grand_salon_left-deflated/acceleration_dataset.pkl'
# dataset_path = '../data/ral2023_dataset/marmotte/boreal_snow/acceleration_dataset.pkl'
# dataset_path = '../data/ral2023_dataset/marmotte/ga_hard_snow_a/acceleration_dataset.pkl'
# dataset_path = '../data/ral2023_dataset/marmotte/ga_hard_snow_b/acceleration_dataset.pkl'
# dataset_path = '../data/ral2023_dataset/marmotte/grand_salon_tile_b/acceleration_dataset.pkl'
# dataset_path = '../data/ral2023_dataset/warthog_tracks/boreal_mud/acceleration_dataset.pkl'
# dataset_path = '../data/ral2023_dataset/warthog_tracks/grand-axe_crusted-snow/acceleration_dataset.pkl'
# dataset_path = '../data/ral2023_dataset/warthog_wheels/gravel_1/acceleration_dataset.pkl'

#dataset_path = '../data/ral2023_dataset/warthog_wheels/ice/acceleration_dataset.pkl'
dataset_path = '../data/ral2023_dataset/warthog_wheels/ice/warthog_wheels_ice_rink_data-raw.pkl'

full_dataset = pd.read_pickle(dataset_path)

### Analysis of the raw dataset obtained from drive
The goal of the folowing cells are to understand the drive dataset. 

In [1]:
print(full_dataset.columns)

NameError: name 'full_dataset' is not defined

Based on the current (E2024) DRIVE, the columns output by the drive logger does not contain the velocity. The velocity is obtained using the scripts in @@@data_utils@@@. More precisely, the parse_dataset.py call the DataParser to produce another dataset. 

## Comments on the first dataparser.

1. The wheel speed comes from the topic : meas_left_vel is it the encoder ?.
2. The diff-drivecompute_diff_drive_body_vels body vel is calculated from the meas_left and meas_right_vel and the ideal diff-drive
3. Why using convolution ?
4. define_calib_quadrans_mask c'est pour faire quoi? Sert a valider que l'ensemble des commandes sont dans un carr/ ?


In [5]:
path = "../data/ral2023_dataset/warthog_wheels/ice/torch_dataset_all.pkl"
df_result_data_parser_1 = pd.read_pickle(path)

ls_columns = df_result_data_parser_1.columns
ls_columns_sans_num = ls_columns.str.slice(start= 0, stop=-3)
print(ls_columns_sans_num.unique())
print(df_result_data_parser_1.head(5))


Index(['init_ic', 'init_icp_r', 'init_icp_pi', 'init_icp_', 'calib_s',
       'cmd_lef', 'cmd_righ', 'cmd_left', 'cmd_right', 'left_wheel_ve',
       'right_wheel_ve', 'left_wheel_vel', 'right_wheel_vel', 'icp_',
       'icp_rol', 'icp_pitc', 'icp_ya', 'icp_x', 'icp_y', 'icp_z', 'icp_roll',
       'icp_pitch', 'icp_yaw', 'imu_ya', 'imu_yaw', 'icp', 'icp_om',
       'steady_state_m', 'transitory_state_m', 'imu_acceleration_',
       'imu_acceleration_x', 'imu_acceleration_y', 'imu_acceleration_z',
       'gt_ic', 'gt_icp_r', 'gt_icp_pi', 'gt_icp_'],
      dtype='object')
   init_icp_x  init_icp_y  init_icp_z  init_icp_roll  init_icp_pitch  \
0         0.0         0.0         0.0            0.0             0.0   
1         0.0         0.0         0.0            0.0             0.0   
2         0.0         0.0         0.0            0.0             0.0   
3         0.0         0.0         0.0            0.0             0.0   
4         0.0         0.0         0.0            0.0           

### Ordre des datasetparser


1. Dans un premier temps, il y a le dataset_parser.py qui va creer un premier dataset g/n/rique appel/ torch_dataset_all (voir le fichier parse_dataset.py). Cette premi'ere /tape impose les deux secondes, calcules les vitesses du corps du v/hicule obtenue par ICP et par les vitesses de roues du v/hicules. 
2. Le parser (slip_dataset_parser) est ensuite appliqu/e sur le r/sultat du premier pour obtenir le dataset pret pour l'entrainement du slip-blr. Concr'etement, il calcule les vitesses de roues avec un mod'ele de premier ordre, calculate the smooth icp algorithm (https://docs.scipy.org/doc/scipy/reference/generated/scipy.interpolate.make_smoothing_spline.html, lambda = 0.8) and calculate the velocity obtained with the resulting speed.  Must understand better the smoothing cause  

### Todo
1. Finir la compr/hension globale du code 
2. Quantifier le bruit en r/gime permanent pour la vitesse d/riv/e de icp 
3. Quantifier le bruit de la vitesse apr'es filtering. 
4. Valider si le mod'ele de premier ordre est une interpolation ou un modele de premier ordre. 